# Kaggle Utilities — Project Ouroboros

This notebook replaces the manual `/kaggle/input/...` copy-and-rename flow. It can:

- load Kaggle secrets safely
- hydrate the persistent Hugging Face cache from the attached `ouroboros-cache` dataset
- install the cached Mamba wheels
- clone/pull the latest repo from GitHub into `/kaggle/working/Ouroboros` and run from the repo root
- inspect local vs Hugging Face checkpoints
- launch Stage 1 with the latest local-or-HF checkpoint selection logic now built into `pretrain.py`
- launch Stage 2 with auto-detected Dual T4 DDP, timeout-safe checkpoint exit, and Stage-2-first resume logic built into `train_sft.py`


In [1]:
import os
import shutil
from pathlib import Path

!cp -r /kaggle/input/datasets/weirdrunner007/ouroboros-cache/* /kaggle/working/

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

In [2]:
from __future__ import annotations

import os

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None


def _normalize_text(value: object | None, *, uppercase: bool = False) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.upper() if uppercase else text


_secret_client = None


def _get_secret_client():
    global _secret_client
    if _secret_client is False:
        return None
    if _secret_client is None:
        if UserSecretsClient is None:
            _secret_client = False
            return None
        try:
            _secret_client = UserSecretsClient()
        except Exception:
            _secret_client = False
            return None
    return _secret_client if _secret_client is not False else None


def _optional_secret(name: str) -> str | None:
    client = _get_secret_client()
    if client is None:
        return None
    try:
        value = client.get_secret(name)
    except Exception:
        return None
    return _normalize_text(value)


def _resolve_value(
    env_names: tuple[str, ...],
    secret_names: tuple[str, ...] = (),
    *,
    uppercase: bool = False,
) -> str | None:
    for env_name in env_names:
        value = _normalize_text(os.environ.get(env_name), uppercase=uppercase)
        if value is not None:
            return value
    for secret_name in secret_names:
        value = _normalize_text(_optional_secret(secret_name), uppercase=uppercase)
        if value is not None:
            return value
    return None


HF_TOKEN = _resolve_value(("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"), ("HF_TOKEN",))
WANDB_KEY = _resolve_value(("WANDB_API_KEY", "WANDB_KEY"), ("WANDB_KEY",))
GITHUB_TOKEN = _resolve_value(("GITHUB_TOKEN", "GH_TOKEN"), ("GITHUB_TOKEN", "GH_TOKEN"))
DILOCO_WORKER_ID = _resolve_value(
    ("DILOCO_WORKER_ID", "OUROBOROS_DILOCO_WORKER_ID", "WORKER_ID"),
    ("DILOCO_WORKER_ID",),
    uppercase=True,
)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    os.environ["WANDB_KEY"] = WANDB_KEY
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    os.environ["GH_TOKEN"] = GITHUB_TOKEN
if DILOCO_WORKER_ID:
    os.environ["DILOCO_WORKER_ID"] = DILOCO_WORKER_ID
    os.environ.setdefault("OUROBOROS_DILOCO_WORKER_ID", DILOCO_WORKER_ID)
    os.environ.setdefault("WORKER_ID", DILOCO_WORKER_ID)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print({
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")),
    "WANDB_KEY": bool(os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_KEY")),
    "GITHUB_TOKEN": bool(os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")),
    "DILOCO_WORKER_ID": bool(
        os.environ.get("DILOCO_WORKER_ID")
        or os.environ.get("OUROBOROS_DILOCO_WORKER_ID")
        or os.environ.get("WORKER_ID")
    ),
})


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = os.environ.get("OUROBOROS_REPO_URL", "https://github.com/deveshpat/Ouroboros.git")
REPO_REF = os.environ.get("OUROBOROS_REPO_REF", "main")
REPO_DIR = Path(os.environ.get("OUROBOROS_REPO_DIR", "/kaggle/working/Ouroboros"))


def run(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True, text=True, capture_output=True)


def authenticated_repo_url(repo_url: str) -> str:
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if not token or not repo_url.startswith("https://") or "github.com" not in repo_url or "@" in repo_url:
        return repo_url
    return f"https://x-access-token:{token}@{repo_url[len('https://'):]}"


def sync_repo(repo_url: str, ref: str, repo_dir: Path) -> str:
    auth_url = authenticated_repo_url(repo_url)
    if not (repo_dir / ".git").exists():
        if repo_dir.exists():
            import shutil
            shutil.rmtree(repo_dir)
        run(["git", "clone", "--branch", ref, auth_url, str(repo_dir)])
    else:
        run(["git", "fetch", "origin", ref], cwd=repo_dir)
        run(["git", "checkout", "-B", ref, "FETCH_HEAD"], cwd=repo_dir)
        run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=repo_dir)
        run(["git", "clean", "-fd"], cwd=repo_dir)
    return run(["git", "rev-parse", "HEAD"], cwd=repo_dir).stdout.strip()


commit = sync_repo(REPO_URL, REPO_REF, REPO_DIR)
os.environ["OUROBOROS_REPO_ROOT"] = str(REPO_DIR)
os.chdir(str(REPO_DIR))
metadata = {
    "repo_ref": REPO_REF,
    "repo_dir": str(REPO_DIR),
    "commit": commit,
    "execution_cwd": os.getcwd(),
    "synced_at_utc": datetime.now(timezone.utc).isoformat(),
}
(Path("/kaggle/working/repo_sync_metadata.json")).write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))


In [ ]:
!torchrun --standalone --nproc_per_node=2 jamba_coconut_finetune.py \
    --data_dir data/coconut_v1 --use_4bit \
    --stage_0_epochs 1 --epochs_per_stage 1 --max_stage 10 \
    --batch_size 4 --grad_accum 8 \
    --val_batch_size 2 \
    --val_skip_buffer_minutes 60 \
    --session_timeout_hours 12.0 --graceful_exit_buffer_minutes 20 \
    --diloco_mode \
    --diloco_worker_id $WORKER_ID \
    --diloco_outer_lr 0.7 \
    --diloco_state_repo WeirdRunner/Ouroboros \
    --diloco_signal_repo deveshpat/Ouroboros \
    --push_to_hub \
    --output_dir runs/diloco \
 #   --output_dir runs/stage3_curriculum \
 #   --wandb_mode offline